# skin-lesion-ai — GPU training (HAM10000)

Runs on Kaggle with **GPU + Internet** enabled. Clones the repo, installs it, prepares the attached dataset, trains, evaluates, and writes results into `artifacts/` (saved as the kernel Output).

In [ ]:
!nvidia-smi -L || echo 'no GPU'

In [ ]:
REPO = 'skin-lesion-ai'
import os, sys, subprocess
if not os.path.isdir(f'/kaggle/working/{REPO}'):
    subprocess.run(['git','clone','--depth','1',
        'https://github.com/sara-tavakoli/'+REPO+'.git'], cwd='/kaggle/working', check=True)
os.chdir(f'/kaggle/working/{REPO}')
# keep Kaggle's GPU torch/torchvision: install our package no-deps + only the extra libs
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e','.'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q', 'pytorch-lightning>=2.4,<2.6', 'timm>=1.0.9', 'torchmetrics>=1.4,<1.8', 'grad-cam>=1.5.4', 'albumentations>=1.4.10', 'omegaconf>=2.3', 'mlflow>=2.14', 'rich>=13.7'], check=True)
SRC = os.path.abspath('src')
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH','')
sys.path.insert(0, SRC)
import torch
import skinlesion; print('skinlesion', getattr(skinlesion, '__version__', 'ok'),
      '| torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 1 · Prepare dataset

In [ ]:
# --- locate or download HAM10000, normalise into data/ham10000/{metadata.csv, images/} ---
import subprocess, pathlib, shutil, os, pandas as pd
INP = pathlib.Path("/kaggle/input")
print("input:", [p.name for p in INP.iterdir()] if INP.exists() else "none")
meta_csv = next((p for p in INP.rglob("HAM10000_metadata*")), None) if INP.exists() else None
if meta_csv is None:
    print("HAM10000 not mounted -> downloading via kaggle CLI")
    dl = pathlib.Path("/kaggle/tmp/ham"); dl.mkdir(parents=True, exist_ok=True)
    subprocess.run(["kaggle","datasets","download","-d","kmader/skin-cancer-mnist-ham10000",
                    "-p",str(dl),"--unzip"], check=True)
    meta_csv = next(p for p in dl.rglob("HAM10000_metadata*"))
SRC = meta_csv.parent
OUT = pathlib.Path("data/ham10000"); (OUT/"images").mkdir(parents=True, exist_ok=True)
meta = pd.read_csv(meta_csv)
meta[[c for c in ["image_id","lesion_id","dx","dx_type","age","sex","localization"] if c in meta.columns]] \
    .to_csv(OUT/"metadata.csv", index=False)
jpegs = {p.stem: p for p in SRC.rglob("*.jpg")}
miss = 0
for iid in meta["image_id"]:
    s = jpegs.get(iid)
    if s is None: miss += 1; continue
    d = OUT/"images"/f"{iid}.jpg"
    if not d.exists():
        try: os.symlink(s, d)
        except OSError: shutil.copy2(s, d)
print("images:", len(list((OUT/'images').glob('*.jpg'))), "| missing:", miss)
print(meta["dx"].value_counts().to_string())

## 2 · Train

In [ ]:
!python scripts/prepare_splits.py --data-dir data/ham10000
!python scripts/train.py --experiment focal_balanced \
    data.image_size=256 data.batch_size=32 data.num_workers=2 \
    train.max_epochs=40 train.precision=16-mixed

In [ ]:
import pathlib
ck = list(pathlib.Path('artifacts').rglob('best.ckpt')) + list(pathlib.Path('artifacts').rglob('*.json'))
assert any(pathlib.Path('artifacts').rglob('best.ckpt')) or any(pathlib.Path('artifacts').rglob('results.json')), \
    'training produced no checkpoint/results - see the log above'
print('train artifacts OK:', [str(p) for p in ck[:6]])


## 3 · Evaluate

In [ ]:
!python scripts/evaluate.py --checkpoint artifacts/best.ckpt --n-bootstrap 2000
!python scripts/explain.py --checkpoint artifacts/best.ckpt --images data/ham10000/images --limit 24 --out artifacts/cams

## 4 · Show results

In [ ]:
import pathlib, IPython.display as D
for md in sorted(pathlib.Path('.').rglob('RESULTS.md')):
    D.display(D.Markdown(md.read_text()))
for png in sorted(pathlib.Path('artifacts').rglob('*.png'))[:16]:
    print(png); D.display(D.Image(str(png)))